DATA PIPELINE

In [3]:
import pandas as pd
import os
import zipfile

print("📦 Rozbaluji obrázky...")
with zipfile.ZipFile('images.zip', 'r') as zip_ref:
    zip_ref.extractall('dataset_images')

extracted_folders = os.listdir('dataset_images')
if 'images' in extracted_folders:
    img_dir = 'dataset_images/images'
else:
    img_dir = 'dataset_images'

print("📊 Načítám tabulková data...")
df = pd.read_csv('brickset_clean.csv')

df['image_path'] = df['set_num'].apply(lambda x: os.path.join(img_dir, f"{x}.jpg"))


df['image_exists'] = df['image_path'].apply(os.path.exists)
df_valid = df[df['image_exists'] == True].copy()

missing_count = len(df) - len(df_valid)

print("\n✅ PŘÍPRAVA DOKONČENA!")
print(f"Celkem spárováno obrázků s daty: {len(df_valid)}")
if missing_count > 0:
    print(f"⚠️ Ignorováno {missing_count} záznamů (obrázek chyběl na disku).")

df_valid[['set_num', 'image_path', 'pieces', 'rrp_price']].head()

📦 Rozbaluji obrázky...
📊 Načítám tabulková data...

✅ PŘÍPRAVA DOKONČENA!
Celkem spárováno obrázků s daty: 806


,set_num,image_path,pieces,rrp_price
0,10312-1,dataset_images/images/10312-1.jpg,2899.0,229.99
1,10313-1,dataset_images/images/10313-1.jpg,939.0,59.99
2,10314-1,dataset_images/images/10314-1.jpg,812.0,49.99
3,10315-1,dataset_images/images/10315-1.jpg,1363.0,109.99
4,10316-1,dataset_images/images/10316-1.jpg,6167.0,499.99


IMPORTY

In [9]:
import cv2
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
import pandas as pd
from tensorflow.keras.layers import Input, Dense, Concatenate, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.optimizers import Adam

VIZUÁLNÍ PŘÍPRAVA DAT

In [10]:
print("🖼️ Načítám a zmenšuji obrázky do paměti...")
IMG_SIZE = 128

images = []
for path in df_valid['image_path']:
    img = cv2.imread(path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    images.append(img)


X_img = np.array(images).astype('float32') / 255.0

🖼️ Načítám a zmenšuji obrázky do paměti...


LOGICKÁ PŘÍPRAVA DAT

In [11]:
print("🔢 Připravuji tabulková data (Normalizace a One-Hot)...")
df_ml = pd.get_dummies(df_valid, columns=['theme'])
tab_features = ['pieces', 'minifigs'] + [c for c in df_ml.columns if c.startswith('theme_')]

X_tab = df_ml[tab_features].values.astype('float32')
y = df_ml['rrp_price'].values.astype('float32')

scaler = MinMaxScaler()
X_tab[:, :2] = scaler.fit_transform(X_tab[:, :2])

X_img_train, X_img_test, X_tab_train, X_tab_test, y_train, y_test = train_test_split(
    X_img, X_tab, y, test_size=0.2, random_state=42
)

🔢 Připravuji tabulková data (Normalizace a One-Hot)...


MULTIMODÁLNÍ SÍŤ

In [12]:
print("🧠 Sestavuji dvouhemisférový AI model...")

image_input = Input(shape=(IMG_SIZE, IMG_SIZE, 3), name="Vstup_Obrazek")
base_model = MobileNetV2(weights='imagenet', include_top=False, input_tensor=image_input)
base_model.trainable = False

x = GlobalAveragePooling2D()(base_model.output)
x = Dense(64, activation='relu')(x)
image_branch = Dropout(0.2)(x)

tab_input = Input(shape=(X_tab.shape[1],), name="Vstup_Data")
y_branch = Dense(64, activation='relu')(tab_input)
tab_branch = Dropout(0.2)(y_branch)

merged = Concatenate()([image_branch, tab_branch])
z = Dense(128, activation='relu')(merged)
z = Dense(64, activation='relu')(z)

output = Dense(1, activation='linear', name="Vystup_Cena")(z)

model = Model(inputs=[image_input, tab_input], outputs=output)
model.compile(optimizer=Adam(learning_rate=0.01), loss='mae')

🧠 Sestavuji dvouhemisférový AI model...


/tmp/ipykernel_1022/1779363.py:4: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base_model = MobileNetV2(weights='imagenet', include_top=False, input_tensor=image_input)


TRÉNINK

In [13]:
print("🚀 STARTUJI TRÉNINK! (Bude to trvat asi 30 epoch/koleček)...")
history = model.fit(
    [X_img_train, X_tab_train], y_train,
    validation_data=([X_img_test, X_tab_test], y_test),
    epochs=30,
    batch_size=32
)
print("🏁 TRÉNINK DOKONČEN!")

🚀 STARTUJI TRÉNINK! (Bude to trvat asi 30 epoch/koleček)...
Epoch 1/30
21/21 ━━━━━━━━━━━━━━━━━━━━ 49s 1s/step - loss: 41.4471 - val_loss: 39.2584
Epoch 2/30
21/21 ━━━━━━━━━━━━━━━━━━━━ 1s 23ms/step - loss: 33.5504 - val_loss: 39.7607
Epoch 3/30
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - loss: 30.6072 - val_loss: 43.0311
Epoch 4/30
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - loss: 28.9045 - val_loss: 35.6666
Epoch 5/30
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - loss: 24.0017 - val_loss: 30.7172
Epoch 6/30
21/21 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - loss: 20.7973 - val_loss: 29.8954
Epoch 7/30
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - loss: 19.9251 - val_loss: 30.0986
Epoch 8/30
21/21 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - loss: 18.6850 - val_loss: 29.8156
Epoch 9/30
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - loss: 16.5853 - val_loss: 27.0982
Epoch 10/30
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - loss: 14.5546 - val_loss: 26.9478
Epoch 11/30
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - loss: 16.0179 - 

DOWNLOAD

In [14]:
import joblib

print("💾 Ukládám multimodální mozek a nástroje...")


model.save('lego_multimodal.keras')
joblib.dump(scaler, 'scaler.pkl')
joblib.dump(tab_features, 'tab_columns.pkl')

print("✅ ÚSPĚCH: Všechny tři soubory jsou bezpečně uloženy na disku Colabu!")
print("👉 Běž vlevo do panelu Soubory a stáhni si je k sobě do počítače.")

💾 Ukládám multimodální mozek a nástroje...
✅ ÚSPĚCH: Všechny tři soubory jsou bezpečně uloženy na disku Colabu!
👉 Běž vlevo do panelu Soubory a stáhni si je k sobě do počítače.
